# Import libraries & define paths

In [50]:
from pathlib import Path
import pandas as pd
import requests

In [51]:
# -----------------------------
# Project paths
# -----------------------------


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

for path in [RAW_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

Project directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project
Raw data directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\raw
Processed data directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed


# 01 Data Collection

This notebook collects the first official datasets for the AI Literacy Gap Index.  
The project starts with Eurostat's regional digital skills dataset because digital skills are the most direct available proxy for AI literacy readiness at NUTS-1 level.

The first goal is to download the Eurostat data inventory, identify the relevant dataset, and save the raw data locally so that later notebooks can work from reproducible local files.

In [58]:
# -----------------------------
# Eurostat inventory download
# -----------------------------

EUROSTAT_INVENTORY_URL = "https://ec.europa.eu/eurostat/api/dissemination/files/inventory?type=data&lang=en"

inventory_path = RAW_DIR / "eurostat_inventory.tsv"

response = requests.get(EUROSTAT_INVENTORY_URL, timeout=60)
response.raise_for_status()

inventory_path.write_bytes(response.content)

eurostat_inventory = pd.read_csv(inventory_path, sep="\t")

print("Inventory shape:", eurostat_inventory.shape)
display(eurostat_inventory.head())
display(eurostat_inventory.columns)

Inventory shape: (8218, 10)


,Code,Type,Source dataset,Last data change,Last structural change,Data download url (tsv),Data download url (csv),Data download url (sdmx),Data structure download url,Open in Data Browser url
0,AACT_ALI01,DATASET,-,2026-05-13T11:00:00+0200,2026-03-24T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
1,AACT_ALI01_R,DATASET,-,2026-03-24T11:00:00+0100,2026-03-24T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
2,AACT_ALI02,DATASET,-,2026-05-13T11:00:00+0200,2026-03-24T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
3,AACT_ALI02_R,DATASET,-,2026-03-24T11:00:00+0100,2026-03-24T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
4,AACT_EAA01,DATASET,-,2026-05-13T11:00:00+0200,2026-03-23T23:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...


Index(['Code', 'Type', 'Source dataset', 'Last data change',
       'Last structural change', 'Data download url (tsv)',
       'Data download url (csv)', 'Data download url (sdmx)',
       'Data structure download url', 'Open in Data Browser url'],
      dtype='str')

## Download core dataset: regional digital skills

The first core dataset for the index is Eurostat's regional digital skills dataset.  
This dataset is used as the main proxy for digital readiness, which is one of the central components of the AI Literacy Gap Index.

Before downloading the data, I check whether the dataset exists in the Eurostat inventory and retrieve its official download URL from there.

In [59]:
# -----------------------------
# Find Eurostat digital skills dataset
# -----------------------------

TARGET_DATASET = "isoc_r_dskl_i"

dataset_match = eurostat_inventory[
    eurostat_inventory["Code"].str.lower() == TARGET_DATASET.lower()
]

if dataset_match.empty:
    raise ValueError(f"Dataset {TARGET_DATASET} was not found in the Eurostat inventory.")

display(dataset_match.T)

# Get official TSV download URL from the inventory
download_url = dataset_match["Data download url (tsv)"].iloc[0]

print("Dataset code:", TARGET_DATASET)
print("Download URL:", download_url)

# -----------------------------
# Download raw dataset
# -----------------------------

raw_dataset_path = RAW_DIR / f"{TARGET_DATASET}.tsv"

response = requests.get(download_url, timeout=120)
response.raise_for_status()

raw_dataset_path.write_bytes(response.content)

print(f"Saved raw dataset to: {raw_dataset_path}")

# Load a first preview
digital_skills_raw = pd.read_csv(raw_dataset_path, sep="\t")

print("Raw dataset shape:", digital_skills_raw.shape)
display(digital_skills_raw.head())
display(digital_skills_raw.columns)

,4289
Code,ISOC_R_DSKL_I
Type,DATASET
Source dataset,-
Last data change,2026-04-17T11:00:00+0200
Last structural change,2026-04-17T11:00:00+0200
Data download url (tsv),https://ec.europa.eu/eurostat/api/disseminatio...
Data download url (csv),https://ec.europa.eu/eurostat/api/disseminatio...
Data download url (sdmx),https://ec.europa.eu/eurostat/api/disseminatio...
Data structure download url,https://ec.europa.eu/eurostat/api/disseminatio...
Open in Data Browser url,https://ec.europa.eu/eurostat/databrowser/prod...


Dataset code: isoc_r_dskl_i
Download URL: https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/ISOC_R_DSKL_I/?format=TSV
Saved raw dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\raw\isoc_r_dskl_i.tsv
Raw dataset shape: (5814, 2)


,"freq,indic_is,unit,geo\TIME_PERIOD",2025
0,"A,I_DSK2_AB,PC_IND,AL",8.07
1,"A,I_DSK2_AB,PC_IND,AT",34.26
2,"A,I_DSK2_AB,PC_IND,AT1",36.81
3,"A,I_DSK2_AB,PC_IND,AT2",31.11
4,"A,I_DSK2_AB,PC_IND,AT3",32.85


Index(['freq,indic_is,unit,geo\TIME_PERIOD', '2025 '], dtype='str')

## Reshape the digital skills dataset

The raw Eurostat file stores several metadata fields in one combined column and the year as a separate value column.  
To make the data usable for analysis, I reshape it into a tidy long format with one row per region, indicator, unit, and year.

In [60]:
# -----------------------------
# Reshape Eurostat compact TSV format
# -----------------------------

# Reload raw data if needed
TARGET_DATASET = "isoc_r_dskl_i"
raw_dataset_path = RAW_DIR / f"{TARGET_DATASET}.tsv"

digital_skills_raw = pd.read_csv(raw_dataset_path, sep="\t")

# Clean column names
digital_skills_raw.columns = digital_skills_raw.columns.str.strip()

# Identify the combined dimension column
dimension_col = [col for col in digital_skills_raw.columns if "\\" in col][0]

# The part before "\TIME_PERIOD" contains the dimension names
dimension_names = dimension_col.split("\\")[0].split(",")

print("Dimension column:", dimension_col)
print("Detected dimensions:", dimension_names)

# Split combined dimension values into separate columns
digital_skills_split = digital_skills_raw[dimension_col].str.split(",", expand=True)
digital_skills_split.columns = dimension_names

# Add year columns
value_cols = [col for col in digital_skills_raw.columns if col != dimension_col]

digital_skills_wide = pd.concat(
    [digital_skills_split, digital_skills_raw[value_cols]],
    axis=1
)

# Reshape from wide to long format
digital_skills_long = digital_skills_wide.melt(
    id_vars=dimension_names,
    value_vars=value_cols,
    var_name="year",
    value_name="value_raw"
)

# Clean year and value columns
digital_skills_long["year"] = digital_skills_long["year"].astype(str).str.strip().astype(int)

# Eurostat values can contain flags after the numeric value.
# This extracts the numeric part and keeps missing values as NaN.
digital_skills_long["value"] = (
    digital_skills_long["value_raw"]
    .astype(str)
    .str.extract(r"([-+]?\d*\.?\d+)")
    .astype(float)
)

# Clean text columns
for col in dimension_names:
    digital_skills_long[col] = digital_skills_long[col].astype(str).str.strip()

# Basic preview
print("Tidy dataset shape:", digital_skills_long.shape)
display(digital_skills_long.head())

print("Available years:")
display(sorted(digital_skills_long["year"].dropna().unique()))

print("Available units:")
display(digital_skills_long["unit"].value_counts())

print("Number of regions:", digital_skills_long["geo"].nunique())
print("Number of indicators:", digital_skills_long["indic_is"].nunique())

Dimension column: freq,indic_is,unit,geo\TIME_PERIOD
Detected dimensions: ['freq', 'indic_is', 'unit', 'geo']
Tidy dataset shape: (5814, 7)


,freq,indic_is,unit,geo,year,value_raw,value
0,A,I_DSK2_AB,PC_IND,AL,2025,8.07,8.07
1,A,I_DSK2_AB,PC_IND,AT,2025,34.26,34.26
2,A,I_DSK2_AB,PC_IND,AT1,2025,36.81,36.81
3,A,I_DSK2_AB,PC_IND,AT2,2025,31.11,31.11
4,A,I_DSK2_AB,PC_IND,AT3,2025,32.85,32.85


Available years:


[np.int64(2025)]

Available units:


unit
PC_IND        2969
PC_IND_IU3    2845
Name: count, dtype: int64

Number of regions: 124
Number of indicators: 24


## Inspect available digital skills indicators

Before selecting variables for the index, I inspect the available digital skills indicators, units, regions, and missing values.  
This step helps separate exploratory understanding from final indicator selection.

In [61]:
# -----------------------------
# Inspect available indicators and data coverage
# -----------------------------

print("Dataset shape:", digital_skills_long.shape)
print("Years:", sorted(digital_skills_long["year"].unique()))
print("Units:", sorted(digital_skills_long["unit"].unique()))
print("Number of geo codes:", digital_skills_long["geo"].nunique())
print("Number of indicators:", digital_skills_long["indic_is"].nunique())

# Indicator-level overview
indicator_overview = (
    digital_skills_long
    .groupby(["indic_is", "unit"], as_index=False)
    .agg(
        n_rows=("value", "size"),
        n_non_missing=("value", "count"),
        n_missing=("value", lambda x: x.isna().sum()),
        min_value=("value", "min"),
        median_value=("value", "median"),
        max_value=("value", "max")
    )
    .sort_values(["indic_is", "unit"])
)

display(indicator_overview)

# Region-level missingness overview
region_overview = (
    digital_skills_long
    .groupby("geo", as_index=False)
    .agg(
        n_rows=("value", "size"),
        n_non_missing=("value", "count"),
        n_missing=("value", lambda x: x.isna().sum())
    )
    .assign(missing_share=lambda df: df["n_missing"] / df["n_rows"])
    .sort_values("missing_share", ascending=False)
)

display(region_overview.head(20))

# Save tidy interim dataset for later notebooks
interim_path = PROCESSED_DIR / "digital_skills_nuts1_tidy.csv"
digital_skills_long.to_csv(interim_path, index=False)

print(f"Saved tidy digital skills dataset to: {interim_path}")

Dataset shape: (5814, 7)
Years: [np.int64(2025)]
Units: ['PC_IND', 'PC_IND_IU3']
Number of geo codes: 124
Number of indicators: 24


,indic_is,unit,n_rows,n_non_missing,n_missing,min_value,median_value,max_value
0,I_DSK2_AB,PC_IND,124,124,0,3.70,27.960,59.42
1,I_DSK2_AB,PC_IND_IU3,124,124,0,4.27,30.650,59.56
2,I_DSK2_B,PC_IND,124,124,0,13.41,28.545,66.44
3,I_DSK2_B,PC_IND_IU3,124,124,0,15.93,30.210,67.48
4,I_DSK2_BAB,PC_IND,124,124,0,18.56,59.570,85.17
5,I_DSK2_BAB,PC_IND_IU3,124,124,0,21.63,63.335,85.36
6,I_DSK2_CC_AB,PC_IND,124,124,0,71.29,85.990,99.42
7,I_DSK2_CC_AB,PC_IND_IU3,124,124,0,83.23,93.600,99.56
8,I_DSK2_CC_B,PC_IND,124,124,0,0.44,4.615,11.97
9,I_DSK2_CC_B,PC_IND_IU3,124,124,0,0.44,4.880,12.94


,geo,n_rows,n_non_missing,n_missing,missing_share
20,DE4,47,45,2,0.042553
23,DE8,47,45,2,0.042553
27,DEE,47,45,2,0.042553
19,DE3,47,45,2,0.042553
24,DE9,47,45,2,0.042553
33,EL4,47,45,2,0.042553
44,FI,47,45,2,0.042553
80,NL2,47,45,2,0.042553
83,NO,47,45,2,0.042553
82,NL4,47,45,2,0.042553


Saved tidy digital skills dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\digital_skills_nuts1_tidy.csv


## Decode Eurostat indicator labels

The raw dataset uses Eurostat technical codes for indicators, units, and regions.  
Before selecting variables for the AI Literacy Gap Index, I decode these codes into readable labels so that indicator choices can be made transparently.

In [62]:
# -----------------------------
# Download Eurostat JSON metadata for labels
# -----------------------------

metadata_url = f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{TARGET_DATASET}"

params = {
    "lang": "en",
    "time": "2025"
}

response = requests.get(metadata_url, params=params, timeout=120)
response.raise_for_status()

digital_skills_metadata = response.json()

print("Metadata keys:", digital_skills_metadata.keys())
print("Available dimensions:", digital_skills_metadata.get("id", []))


# -----------------------------
# Helper function to extract dimension labels
# -----------------------------
def extract_dimension_labels(metadata, dimension_name):
    """
    Extracts Eurostat code-label mappings from the JSON metadata.
    """
    dimension = metadata["dimension"][dimension_name]
    category = dimension["category"]

    labels = category.get("label", {})
    index = category.get("index", {})

    # Prefer index keys because they define the available codes
    codes = list(index.keys()) if isinstance(index, dict) else list(labels.keys())

    label_table = pd.DataFrame({
        dimension_name: codes,
        f"{dimension_name}_label": [labels.get(code, code) for code in codes]
    })

    return label_table


# -----------------------------
# Extract labels for relevant dimensions
# -----------------------------

indic_labels = extract_dimension_labels(digital_skills_metadata, "indic_is")
unit_labels = extract_dimension_labels(digital_skills_metadata, "unit")
geo_labels = extract_dimension_labels(digital_skills_metadata, "geo")

print("Indicator labels:")
display(indic_labels)

print("Unit labels:")
display(unit_labels)

print("Geo labels preview:")
display(geo_labels.head(20))


# -----------------------------
# Merge labels into tidy dataset
# -----------------------------

digital_skills_labeled = (
    digital_skills_long
    .merge(indic_labels, on="indic_is", how="left")
    .merge(unit_labels, on="unit", how="left")
    .merge(geo_labels, on="geo", how="left")
)

display(digital_skills_labeled.head())

# Save labeled dataset
labeled_path = PROCESSED_DIR / "digital_skills_nuts1_labeled.csv"
digital_skills_labeled.to_csv(labeled_path, index=False)

print(f"Saved labeled digital skills dataset to: {labeled_path}")

Metadata keys: dict_keys(['version', 'class', 'label', 'source', 'updated', 'value', 'status', 'id', 'size', 'dimension', 'extension'])
Available dimensions: ['freq', 'indic_is', 'unit', 'geo', 'time']
Indicator labels:


,indic_is,indic_is_label
0,I_DSK2_IC_S,Individuals with online information and commun...
1,I_DSK2_DCC_BAB,Individuals with basic or above basic digital ...
2,I_DSK2_DCC_AB,Individuals with above basic digital content c...
3,I_DSK2_DCC_B,Individuals with basic digital content creatio...
4,I_DSK2_SF_BAB,Individuals with basic or above basic safety s...
5,I_DSK2_SF_AB,Individuals with above basic safety skills
6,I_DSK2_SF_B,Individuals with basic safety skills
7,I_DSK2_PS_BAB,Individuals with basic or above basic problem ...
8,I_DSK2_PS_AB,Individuals with above basic problem solving s...
9,I_DSK2_PS_B,Individuals with basic problem solving skills


Unit labels:


,unit,unit_label
0,PC_IND,Percentage of individuals
1,PC_IND_IU3,Percentage of individuals who used internet in...


Geo labels preview:


,geo,geo_label
0,BE,Belgium
1,BE1,Région de Bruxelles-Capitale/Brussels Hoofdste...
2,BE2,Vlaams Gewest
3,BE3,Région wallonne
4,BG,Bulgaria
5,BG3,Severna i Yugoiztochna Bulgaria
6,BG4,Yugozapadna i Yuzhna tsentralna Bulgaria
7,CZ,Czechia
8,DK,Denmark
9,DE,Germany


,freq,indic_is,unit,geo,year,value_raw,value,indic_is_label,unit_label,geo_label
0,A,I_DSK2_AB,PC_IND,AL,2025,8.07,8.07,Individuals with above basic overall digital s...,Percentage of individuals,Albania
1,A,I_DSK2_AB,PC_IND,AT,2025,34.26,34.26,Individuals with above basic overall digital s...,Percentage of individuals,Austria
2,A,I_DSK2_AB,PC_IND,AT1,2025,36.81,36.81,Individuals with above basic overall digital s...,Percentage of individuals,Ostösterreich
3,A,I_DSK2_AB,PC_IND,AT2,2025,31.11,31.11,Individuals with above basic overall digital s...,Percentage of individuals,Südösterreich
4,A,I_DSK2_AB,PC_IND,AT3,2025,32.85,32.85,Individuals with above basic overall digital s...,Percentage of individuals,Westösterreich


Saved labeled digital skills dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\digital_skills_nuts1_labeled.csv


## Filter to official NUTS-1 regions

The Eurostat digital skills dataset contains both country-level and regional geo codes.  
For this project, the unit of analysis is NUTS-1 regions, so I match the dataset against the official NUTS-1 classification and keep only valid NUTS-1 geo codes.

This avoids relying on simple code-length rules, because some countries can also be represented as a single NUTS-1 region.

In [63]:
# -----------------------------
# Download official NUTS-1 classification from GISCO
# -----------------------------

import json

NUTS1_GEOJSON_URL = (
    "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/"
    "NUTS_RG_01M_2024_4326_LEVL_1.geojson"
)

nuts1_geojson_path = RAW_DIR / "NUTS_RG_01M_2024_4326_LEVL_1.geojson"

response = requests.get(NUTS1_GEOJSON_URL, timeout=120)
response.raise_for_status()

nuts1_geojson_path.write_bytes(response.content)

with open(nuts1_geojson_path, "r", encoding="utf-8") as f:
    nuts1_geojson = json.load(f)

# Extract NUTS-1 metadata from GeoJSON properties
nuts1_lookup = pd.DataFrame([
    feature["properties"]
    for feature in nuts1_geojson["features"]
])

# Keep only relevant columns if available
available_cols = [col for col in ["NUTS_ID", "NAME_LATN", "CNTR_CODE", "LEVL_CODE"] if col in nuts1_lookup.columns]
nuts1_lookup = nuts1_lookup[available_cols].drop_duplicates()

nuts1_lookup = nuts1_lookup.rename(columns={
    "NUTS_ID": "geo",
    "NAME_LATN": "nuts1_name",
    "CNTR_CODE": "country_code",
    "LEVL_CODE": "nuts_level"
})

print("Official NUTS-1 regions:", nuts1_lookup["geo"].nunique())
display(nuts1_lookup.head())


# -----------------------------
# Filter digital skills data to official NUTS-1 regions
# -----------------------------

digital_skills_nuts1 = digital_skills_labeled.merge(
    nuts1_lookup,
    on="geo",
    how="inner"
)

excluded_geo_codes = sorted(
    set(digital_skills_labeled["geo"].unique()) - set(digital_skills_nuts1["geo"].unique())
)

print("Original geo codes:", digital_skills_labeled["geo"].nunique())
print("Matched NUTS-1 geo codes:", digital_skills_nuts1["geo"].nunique())
print("Excluded geo codes:", len(excluded_geo_codes))
print("Examples of excluded geo codes:")
display(excluded_geo_codes[:30])

print("Filtered NUTS-1 dataset shape:", digital_skills_nuts1.shape)
display(digital_skills_nuts1.head())


# -----------------------------
# Save filtered NUTS-1 dataset
# -----------------------------

nuts1_path = PROCESSED_DIR / "digital_skills_nuts1_labeled_filtered.csv"
digital_skills_nuts1.to_csv(nuts1_path, index=False)

nuts1_lookup_path = PROCESSED_DIR / "nuts1_lookup.csv"
nuts1_lookup.to_csv(nuts1_lookup_path, index=False)

print(f"Saved filtered NUTS-1 digital skills dataset to: {nuts1_path}")
print(f"Saved NUTS-1 lookup table to: {nuts1_lookup_path}")

Official NUTS-1 regions: 115


,geo,nuts1_name,country_code,nuts_level
0,DEF,Schleswig-Holstein,DE,1
1,DE2,Bayern,DE,1
2,DE3,Berlin,DE,1
3,DEG,Thüringen,DE,1
4,EL4,"Nisia Aigaiou, Kriti",EL,1


Original geo codes: 124
Matched NUTS-1 geo codes: 88
Excluded geo codes: 36
Examples of excluded geo codes:


['AL',
 'AT',
 'BA',
 'BE',
 'BG',
 'CH',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'EL',
 'ES',
 'FI',
 'FR',
 'HR',
 'HU',
 'IE',
 'IT',
 'LT',
 'LU',
 'LV',
 'ME',
 'MK',
 'MT',
 'NL',
 'NO',
 'PL',
 'PT',
 'RO']

Filtered NUTS-1 dataset shape: (4126, 13)


,freq,indic_is,unit,geo,year,value_raw,value,indic_is_label,unit_label,geo_label,nuts1_name,country_code,nuts_level
0,A,I_DSK2_AB,PC_IND,AT1,2025,36.81,36.81,Individuals with above basic overall digital s...,Percentage of individuals,Ostösterreich,Ostösterreich,AT,1
1,A,I_DSK2_AB,PC_IND,AT2,2025,31.11,31.11,Individuals with above basic overall digital s...,Percentage of individuals,Südösterreich,Südösterreich,AT,1
2,A,I_DSK2_AB,PC_IND,AT3,2025,32.85,32.85,Individuals with above basic overall digital s...,Percentage of individuals,Westösterreich,Westösterreich,AT,1
3,A,I_DSK2_AB,PC_IND,BE1,2025,35.82,35.82,Individuals with above basic overall digital s...,Percentage of individuals,Région de Bruxelles-Capitale/Brussels Hoofdste...,Région de Bruxelles-Capitale/Brussels Hoofdste...,BE,1
4,A,I_DSK2_AB,PC_IND,BE2,2025,29.59,29.59,Individuals with above basic overall digital s...,Percentage of individuals,Vlaams Gewest,Vlaams Gewest,BE,1


Saved filtered NUTS-1 digital skills dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\digital_skills_nuts1_labeled_filtered.csv
Saved NUTS-1 lookup table to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\nuts1_lookup.csv


## Inspect available NUTS-1 digital skills indicators

After filtering the data to official NUTS-1 regions, I inspect the readable indicator labels and their coverage.  
This step is exploratory and does not yet define the final index variables. The goal is to understand which indicators are available, complete, and conceptually useful for the AI Literacy Gap Index.

In [64]:
# -----------------------------
# Load filtered NUTS-1 data if needed
# -----------------------------

nuts1_path = PROCESSED_DIR / "digital_skills_nuts1_labeled_filtered.csv"

if "digital_skills_nuts1" not in globals():
    digital_skills_nuts1 = pd.read_csv(nuts1_path)

print("NUTS-1 digital skills dataset shape:", digital_skills_nuts1.shape)
print("Number of NUTS-1 regions:", digital_skills_nuts1["geo"].nunique())
print("Available years:", sorted(digital_skills_nuts1["year"].unique()))
print("Available units:", sorted(digital_skills_nuts1["unit"].unique()))

# -----------------------------
# Indicator overview with labels
# -----------------------------

indicator_label_overview = (
    digital_skills_nuts1
    .groupby(["indic_is", "indic_is_label", "unit", "unit_label"], as_index=False)
    .agg(
        n_regions=("geo", "nunique"),
        n_rows=("value", "size"),
        n_non_missing=("value", "count"),
        n_missing=("value", lambda x: x.isna().sum()),
        min_value=("value", "min"),
        median_value=("value", "median"),
        max_value=("value", "max")
    )
    .assign(
        missing_share=lambda df: df["n_missing"] / df["n_rows"]
    )
    .sort_values(["unit", "indic_is"])
)

display(indicator_label_overview)

# Show the indicator labels more compactly
compact_indicator_list = (
    indicator_label_overview[
        ["indic_is", "indic_is_label", "unit", "unit_label", "n_regions", "missing_share"]
    ]
    .drop_duplicates()
    .sort_values(["unit", "indic_is"])
)

display(compact_indicator_list)

# Save overview for documentation
indicator_overview_path = PROCESSED_DIR / "digital_skills_indicator_overview.csv"
indicator_label_overview.to_csv(indicator_overview_path, index=False)

print(f"Saved indicator overview to: {indicator_overview_path}")

NUTS-1 digital skills dataset shape: (4126, 13)
Number of NUTS-1 regions: 88
Available years: [np.int64(2025)]
Available units: ['PC_IND', 'PC_IND_IU3']


,indic_is,indic_is_label,unit,unit_label,n_regions,n_rows,n_non_missing,n_missing,min_value,median_value,max_value,missing_share
0,I_DSK2_AB,Individuals with above basic overall digital s...,PC_IND,Percentage of individuals,88,88,88,0,3.70,27.960,59.42,0.000000
2,I_DSK2_B,Individuals with basic overall digital skills ...,PC_IND,Percentage of individuals,88,88,88,0,13.41,28.630,39.55,0.000000
4,I_DSK2_BAB,Individuals with basic or above basic overall ...,PC_IND,Percentage of individuals,88,88,88,0,18.56,59.645,85.17,0.000000
6,I_DSK2_CC_AB,Individuals with above basic communication and...,PC_IND,Percentage of individuals,88,88,88,0,71.29,85.515,99.42,0.000000
8,I_DSK2_CC_B,Individuals with basic communication and colla...,PC_IND,Percentage of individuals,88,88,88,0,0.44,4.855,10.95,0.000000
10,I_DSK2_CC_BAB,Individuals with basic or above basic communic...,PC_IND,Percentage of individuals,88,88,88,0,78.31,92.070,99.86,0.000000
12,I_DSK2_DCC_AB,Individuals with above basic digital content c...,PC_IND,Percentage of individuals,88,88,88,0,6.98,47.115,71.40,0.000000
14,I_DSK2_DCC_B,Individuals with basic digital content creatio...,PC_IND,Percentage of individuals,88,88,88,0,10.20,21.545,35.22,0.000000
16,I_DSK2_DCC_BAB,Individuals with basic or above basic digital ...,PC_IND,Percentage of individuals,88,88,88,0,25.59,69.855,89.26,0.000000
18,I_DSK2_IC_S,Individuals with online information and commun...,PC_IND,Percentage of individuals,85,85,73,12,0.00,2.540,9.27,0.141176


,indic_is,indic_is_label,unit,unit_label,n_regions,missing_share
0,I_DSK2_AB,Individuals with above basic overall digital s...,PC_IND,Percentage of individuals,88,0.000000
2,I_DSK2_B,Individuals with basic overall digital skills ...,PC_IND,Percentage of individuals,88,0.000000
4,I_DSK2_BAB,Individuals with basic or above basic overall ...,PC_IND,Percentage of individuals,88,0.000000
6,I_DSK2_CC_AB,Individuals with above basic communication and...,PC_IND,Percentage of individuals,88,0.000000
8,I_DSK2_CC_B,Individuals with basic communication and colla...,PC_IND,Percentage of individuals,88,0.000000
10,I_DSK2_CC_BAB,Individuals with basic or above basic communic...,PC_IND,Percentage of individuals,88,0.000000
12,I_DSK2_DCC_AB,Individuals with above basic digital content c...,PC_IND,Percentage of individuals,88,0.000000
14,I_DSK2_DCC_B,Individuals with basic digital content creatio...,PC_IND,Percentage of individuals,88,0.000000
16,I_DSK2_DCC_BAB,Individuals with basic or above basic digital ...,PC_IND,Percentage of individuals,88,0.000000
18,I_DSK2_IC_S,Individuals with online information and commun...,PC_IND,Percentage of individuals,85,0.141176


Saved indicator overview to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\digital_skills_indicator_overview.csv


## Define candidate datasets for the AI Literacy Gap Index

After preparing the first digital skills dataset, I define a candidate data source registry for the full AI Literacy Gap Index.  
The index should not rely only on digital skills, because AI literacy gap risk is also shaped by education, adult learning, social vulnerability, demographic structure, and regional AI exposure.

This step checks whether the selected Eurostat dataset codes are available in the official Eurostat inventory before downloading them.

In [65]:
# -----------------------------
# Candidate Eurostat datasets for the AI Literacy Gap Index
# -----------------------------

dataset_candidates = pd.DataFrame([
    {
        "dataset_code": "isoc_r_dskl_i",
        "pillar": "Digital readiness",
        "intended_use": "Core proxy for regional digital skills at NUTS-1 level",
        "expected_geo_level": "NUTS-1",
        "priority": "Core"
    },
    {
        "dataset_code": "tgs00107",
        "pillar": "Social vulnerability",
        "intended_use": "People at risk of poverty or social exclusion by region",
        "expected_geo_level": "NUTS-2",
        "priority": "Core"
    },
    {
        "dataset_code": "edat_lfse_04",
        "pillar": "Education",
        "intended_use": "Educational attainment by region",
        "expected_geo_level": "NUTS-2",
        "priority": "Core"
    },
    {
        "dataset_code": "trng_lfse_04",
        "pillar": "Adult learning",
        "intended_use": "Participation in education and training by region",
        "expected_geo_level": "NUTS-2",
        "priority": "Core"
    },
    {
        "dataset_code": "demo_r_d2jan",
        "pillar": "Demographics",
        "intended_use": "Population by age, sex, and region",
        "expected_geo_level": "NUTS-2",
        "priority": "Core"
    },
    {
        "dataset_code": "lfst_r_lfu3pers",
        "pillar": "Labour market vulnerability",
        "intended_use": "Unemployment by education level and region",
        "expected_geo_level": "NUTS-2",
        "priority": "Optional core"
    },
    {
        "dataset_code": "isoc_r_eb_ain2",
        "pillar": "AI exposure",
        "intended_use": "Enterprise AI adoption by region",
        "expected_geo_level": "NUTS-2",
        "priority": "Core if available"
    }
])

# -----------------------------
# Match candidates against Eurostat inventory
# -----------------------------

inventory_lookup = eurostat_inventory.copy()
inventory_lookup["code_lower"] = inventory_lookup["Code"].str.lower()
dataset_candidates["code_lower"] = dataset_candidates["dataset_code"].str.lower()

dataset_registry = dataset_candidates.merge(
    inventory_lookup,
    on="code_lower",
    how="left"
)

dataset_registry["found_in_inventory"] = dataset_registry["Code"].notna()

# Keep useful columns
registry_cols = [
    "dataset_code",
    "pillar",
    "intended_use",
    "expected_geo_level",
    "priority",
    "found_in_inventory",
    "Code",
    "Last data change",
    "Last structural change",
    "Data download url (tsv)",
    "Open in Data Browser url"
]

dataset_registry = dataset_registry[registry_cols]

display(dataset_registry)

# Save registry for project documentation
registry_path = PROCESSED_DIR / "data_source_registry.csv"
dataset_registry.to_csv(registry_path, index=False)

print(f"Saved data source registry to: {registry_path}")

# Quick warning if any candidate datasets were not found
missing_candidates = dataset_registry.loc[
    ~dataset_registry["found_in_inventory"],
    ["dataset_code", "pillar", "intended_use"]
]

if len(missing_candidates) > 0:
    print("Datasets not found in inventory:")
    display(missing_candidates)
else:
    print("All candidate datasets were found in the Eurostat inventory.")

,dataset_code,pillar,intended_use,expected_geo_level,priority,found_in_inventory,Code,Last data change,Last structural change,Data download url (tsv),Open in Data Browser url
0,isoc_r_dskl_i,Digital readiness,Core proxy for regional digital skills at NUTS...,NUTS-1,Core,True,ISOC_R_DSKL_I,2026-04-17T11:00:00+0200,2026-04-17T11:00:00+0200,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
1,tgs00107,Social vulnerability,People at risk of poverty or social exclusion ...,NUTS-2,Core,True,TGS00107,2026-05-06T23:00:00+0200,2025-12-22T23:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
2,edat_lfse_04,Education,Educational attainment by region,NUTS-2,Core,True,EDAT_LFSE_04,2026-04-16T23:00:00+0200,2026-04-16T23:00:00+0200,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
3,trng_lfse_04,Adult learning,Participation in education and training by region,NUTS-2,Core,True,TRNG_LFSE_04,2026-04-16T23:00:00+0200,2026-04-16T23:00:00+0200,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
4,demo_r_d2jan,Demographics,"Population by age, sex, and region",NUTS-2,Core,True,DEMO_R_D2JAN,2026-05-08T11:00:00+0200,2026-01-20T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
5,lfst_r_lfu3pers,Labour market vulnerability,Unemployment by education level and region,NUTS-2,Optional core,True,LFST_R_LFU3PERS,2026-04-17T11:00:00+0200,2026-04-17T11:00:00+0200,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
6,isoc_r_eb_ain2,AI exposure,Enterprise AI adoption by region,NUTS-2,Core if available,True,ISOC_R_EB_AIN2,2026-02-27T11:00:00+0100,2026-01-26T23:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...


Saved data source registry to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\data_source_registry.csv
All candidate datasets were found in the Eurostat inventory.


## Download social vulnerability data

To connect the AI Literacy Gap Index to SDG 10, I add a regional social vulnerability indicator.  
The first dataset used for this pillar is the share of people at risk of poverty or social exclusion by region.

This indicator helps identify regions where low AI readiness could reinforce existing socioeconomic disadvantage.

In [66]:
# -----------------------------
# Download Eurostat poverty / social exclusion dataset
# -----------------------------

TARGET_DATASET = "tgs00107"

# Find dataset in Eurostat inventory
dataset_match = eurostat_inventory[
    eurostat_inventory["Code"].str.lower() == TARGET_DATASET.lower()
]

if dataset_match.empty:
    raise ValueError(f"Dataset {TARGET_DATASET} was not found in the Eurostat inventory.")

display(dataset_match.T)

download_url = dataset_match["Data download url (tsv)"].iloc[0]

print("Dataset code:", TARGET_DATASET)
print("Download URL:", download_url)

# Download raw dataset
raw_dataset_path = RAW_DIR / f"{TARGET_DATASET}.tsv"

response = requests.get(download_url, timeout=120)
response.raise_for_status()

raw_dataset_path.write_bytes(response.content)

print(f"Saved raw dataset to: {raw_dataset_path}")

# Load first preview
poverty_raw = pd.read_csv(raw_dataset_path, sep="\t")

poverty_raw.columns = poverty_raw.columns.str.strip()

print("Raw dataset shape:", poverty_raw.shape)
display(poverty_raw.head())
display(poverty_raw.columns)

,7302
Code,TGS00107
Type,DATASET
Source dataset,-
Last data change,2026-05-06T23:00:00+0200
Last structural change,2025-12-22T23:00:00+0100
Data download url (tsv),https://ec.europa.eu/eurostat/api/disseminatio...
Data download url (csv),https://ec.europa.eu/eurostat/api/disseminatio...
Data download url (sdmx),https://ec.europa.eu/eurostat/api/disseminatio...
Data structure download url,https://ec.europa.eu/eurostat/api/disseminatio...
Open in Data Browser url,https://ec.europa.eu/eurostat/databrowser/prod...


Dataset code: tgs00107
Download URL: https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/TGS00107/?format=TSV
Saved raw dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\raw\tgs00107.tsv
Raw dataset shape: (269, 12)


,"freq,unit,geo\TIME_PERIOD",2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,"A,PC_POP,AL01",:,:,62.0,54.3,55.4,45.6,46.7,47.1,45.0,:,:
1,"A,PC_POP,AL02",:,:,56.6,54.3,49.2,45.4,46.6,44.1,41.7,:,:
2,"A,PC_POP,AL03",:,:,57.7,53.2,48.4,47.8,46.5,42.6,39.8,:,:
3,"A,PC_POP,AT11",:,:,:,:,:,:,13.0,7.3,12.3,10.0,11.7
4,"A,PC_POP,AT12",:,:,:,:,:,:,15.2,16.2,13.2,12.4,13.7


Index(['freq,unit,geo\TIME_PERIOD', '2015', '2016', '2017', '2018', '2019',
       '2020', '2021', '2022', '2023', '2024', '2025'],
      dtype='str')

## Reshape poverty and social exclusion data

The poverty and social exclusion dataset is stored in Eurostat's wide format, with years as separate columns.  
I reshape the data into a tidy long format so that each row represents one region-year observation. This makes the dataset easier to inspect, filter, aggregate, and later merge with the AI Literacy Gap Index dataset.

In [67]:
# -----------------------------
# Reshape poverty / social exclusion dataset
# -----------------------------

TARGET_DATASET = "tgs00107"
raw_dataset_path = RAW_DIR / f"{TARGET_DATASET}.tsv"

# Reload raw data if needed
poverty_raw = pd.read_csv(raw_dataset_path, sep="\t")
poverty_raw.columns = poverty_raw.columns.str.strip()

# Identify Eurostat combined dimension column
dimension_col = [col for col in poverty_raw.columns if "\\" in col][0]
dimension_names = dimension_col.split("\\")[0].split(",")

print("Dimension column:", dimension_col)
print("Detected dimensions:", dimension_names)

# Split combined dimension values into separate columns
poverty_split = poverty_raw[dimension_col].str.split(",", expand=True)
poverty_split.columns = dimension_names

# Identify year columns
value_cols = [col for col in poverty_raw.columns if col != dimension_col]

# Combine dimensions and year values
poverty_wide = pd.concat(
    [poverty_split, poverty_raw[value_cols]],
    axis=1
)

# Reshape from wide to long format
poverty_long = poverty_wide.melt(
    id_vars=dimension_names,
    value_vars=value_cols,
    var_name="year",
    value_name="value_raw"
)

# Clean year
poverty_long["year"] = poverty_long["year"].astype(str).str.strip().astype(int)

# Extract numeric values.
# Eurostat missing values such as ":" become NaN.
poverty_long["value"] = (
    poverty_long["value_raw"]
    .astype(str)
    .str.extract(r"([-+]?\d*\.?\d+)")
    .astype(float)
)

# Clean text columns
for col in dimension_names:
    poverty_long[col] = poverty_long[col].astype(str).str.strip()

# Basic inspection
print("Tidy poverty dataset shape:", poverty_long.shape)
display(poverty_long.head())

print("Available years:")
display(sorted(poverty_long["year"].dropna().unique()))

print("Available units:")
display(poverty_long["unit"].value_counts())

print("Number of geo codes:", poverty_long["geo"].nunique())

print("Missing value summary:")
display(
    poverty_long
    .groupby("year", as_index=False)
    .agg(
        n_rows=("value", "size"),
        n_non_missing=("value", "count"),
        n_missing=("value", lambda x: x.isna().sum()),
        missing_share=("value", lambda x: x.isna().mean())
    )
)

# Save tidy dataset
poverty_tidy_path = PROCESSED_DIR / "poverty_social_exclusion_tidy.csv"
poverty_long.to_csv(poverty_tidy_path, index=False)

print(f"Saved tidy poverty dataset to: {poverty_tidy_path}")

Dimension column: freq,unit,geo\TIME_PERIOD
Detected dimensions: ['freq', 'unit', 'geo']
Tidy poverty dataset shape: (2959, 6)


,freq,unit,geo,year,value_raw,value
0,A,PC_POP,AL01,2015,:,NaN
1,A,PC_POP,AL02,2015,:,NaN
2,A,PC_POP,AL03,2015,:,NaN
3,A,PC_POP,AT11,2015,:,NaN
4,A,PC_POP,AT12,2015,:,NaN


Available years:


[np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2024),
 np.int64(2025)]

Available units:


unit
PC_POP    2959
Name: count, dtype: int64

Number of geo codes: 269
Missing value summary:


,year,n_rows,n_non_missing,n_missing,missing_share
0,2015,269,107,162,0.602230
1,2016,269,122,147,0.546468
2,2017,269,127,142,0.527881
3,2018,269,148,121,0.449814
4,2019,269,169,100,0.371747
5,2020,269,168,101,0.375465
6,2021,269,228,41,0.152416
7,2022,269,254,15,0.055762
8,2023,269,250,19,0.070632
9,2024,269,253,16,0.059480


Saved tidy poverty dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\poverty_social_exclusion_tidy.csv


## Inspect regional level of poverty data

The poverty and social exclusion indicator is needed for the social vulnerability pillar of the AI Literacy Gap Index.  
Before merging it with the NUTS-1 digital skills data, I inspect which regional levels are available.

Because the indicator is a percentage of the population, NUTS-2 values should not be averaged directly into NUTS-1 regions. A later aggregation step should use population weights.

In [68]:
# -----------------------------
# Inspect geo levels in poverty data
# -----------------------------
from pathlib import Path
import pandas as pd
import requests
import json

# Reload poverty tidy dataset if needed
poverty_tidy_path = PROCESSED_DIR / "poverty_social_exclusion_tidy.csv"

if "poverty_long" not in globals():
    poverty_long = pd.read_csv(poverty_tidy_path)

# Reload NUTS-1 lookup if needed
nuts1_lookup_path = PROCESSED_DIR / "nuts1_lookup.csv"

if "nuts1_lookup" not in globals():
    nuts1_lookup = pd.read_csv(nuts1_lookup_path)

# -----------------------------
# Download official NUTS-2 classification
# -----------------------------
NUTS2_GEOJSON_URL = (
    "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/"
    "NUTS_RG_01M_2024_4326_LEVL_2.geojson"
)

nuts2_geojson_path = RAW_DIR / "NUTS_RG_01M_2024_4326_LEVL_2.geojson"

if not nuts2_geojson_path.exists():
    response = requests.get(NUTS2_GEOJSON_URL, timeout=120)
    response.raise_for_status()
    nuts2_geojson_path.write_bytes(response.content)

with open(nuts2_geojson_path, "r", encoding="utf-8") as f:
    nuts2_geojson = json.load(f)

nuts2_lookup = pd.DataFrame([
    feature["properties"]
    for feature in nuts2_geojson["features"]
])

available_cols = [col for col in ["NUTS_ID", "NAME_LATN", "CNTR_CODE", "LEVL_CODE"] if col in nuts2_lookup.columns]
nuts2_lookup = nuts2_lookup[available_cols].drop_duplicates()

nuts2_lookup = nuts2_lookup.rename(columns={
    "NUTS_ID": "geo",
    "NAME_LATN": "nuts2_name",
    "CNTR_CODE": "country_code",
    "LEVL_CODE": "nuts_level"
})

# Derive parent NUTS-1 code from NUTS-2 code
nuts2_lookup["parent_nuts1"] = nuts2_lookup["geo"].str[:3]

# Add parent NUTS-1 name
nuts2_to_nuts1 = nuts2_lookup.merge(
    nuts1_lookup[["geo", "nuts1_name"]].rename(columns={
        "geo": "parent_nuts1",
        "nuts1_name": "parent_nuts1_name"
    }),
    on="parent_nuts1",
    how="left"
)

print("Official NUTS-1 regions:", nuts1_lookup["geo"].nunique())
print("Official NUTS-2 regions:", nuts2_lookup["geo"].nunique())

# -----------------------------
# Classify poverty geo codes by available NUTS level
# -----------------------------
poverty_geo = pd.DataFrame({
    "geo": sorted(poverty_long["geo"].unique())
})

poverty_geo["is_nuts1"] = poverty_geo["geo"].isin(nuts1_lookup["geo"])
poverty_geo["is_nuts2"] = poverty_geo["geo"].isin(nuts2_lookup["geo"])

def classify_geo_level(row):
    if row["is_nuts1"]:
        return "NUTS-1"
    if row["is_nuts2"]:
        return "NUTS-2"
    if len(row["geo"]) == 2:
        return "Country or non-NUTS country code"
    return "Other / unmatched"

poverty_geo["geo_level_detected"] = poverty_geo.apply(classify_geo_level, axis=1)

display(
    poverty_geo["geo_level_detected"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "geo_level_detected", "geo_level_detected": "n_geo_codes"})
)

display(poverty_geo.head(30))

# -----------------------------
# Attach NUTS-2 to NUTS-1 mapping where possible
# -----------------------------
poverty_with_mapping = poverty_long.merge(
    nuts2_to_nuts1[["geo", "nuts2_name", "parent_nuts1", "parent_nuts1_name", "country_code"]],
    on="geo",
    how="left"
)

print("Poverty rows:", poverty_with_mapping.shape[0])
print("Rows matched to NUTS-2:", poverty_with_mapping["parent_nuts1"].notna().sum())
print("Rows not matched to NUTS-2:", poverty_with_mapping["parent_nuts1"].isna().sum())

display(poverty_with_mapping.head())

# Save mapping tables
nuts2_lookup_path = PROCESSED_DIR / "nuts2_lookup.csv"
nuts2_to_nuts1_path = PROCESSED_DIR / "nuts2_to_nuts1_lookup.csv"
poverty_mapped_path = PROCESSED_DIR / "poverty_social_exclusion_mapped.csv"

nuts2_lookup.to_csv(nuts2_lookup_path, index=False)
nuts2_to_nuts1.to_csv(nuts2_to_nuts1_path, index=False)
poverty_with_mapping.to_csv(poverty_mapped_path, index=False)

print(f"Saved NUTS-2 lookup to: {nuts2_lookup_path}")
print(f"Saved NUTS-2 to NUTS-1 lookup to: {nuts2_to_nuts1_path}")
print(f"Saved mapped poverty dataset to: {poverty_mapped_path}")

Official NUTS-1 regions: 115
Official NUTS-2 regions: 299


,n_geo_codes,count
0,NUTS-2,256
1,Other / unmatched,13


,geo,is_nuts1,is_nuts2,geo_level_detected
0,AL01,False,True,NUTS-2
1,AL02,False,True,NUTS-2
2,AL03,False,True,NUTS-2
3,AT11,False,True,NUTS-2
4,AT12,False,True,NUTS-2
5,AT13,False,True,NUTS-2
6,AT21,False,True,NUTS-2
7,AT22,False,True,NUTS-2
8,AT31,False,True,NUTS-2
9,AT32,False,True,NUTS-2


Poverty rows: 2959
Rows matched to NUTS-2: 2816
Rows not matched to NUTS-2: 143


,freq,unit,geo,year,value_raw,value,nuts2_name,parent_nuts1,parent_nuts1_name,country_code
0,A,PC_POP,AL01,2015,:,NaN,Veri,AL0,Shqipëria,AL
1,A,PC_POP,AL02,2015,:,NaN,Qender,AL0,Shqipëria,AL
2,A,PC_POP,AL03,2015,:,NaN,Jug,AL0,Shqipëria,AL
3,A,PC_POP,AT11,2015,:,NaN,Burgenland,AT1,Ostösterreich,AT
4,A,PC_POP,AT12,2015,:,NaN,Niederösterreich,AT1,Ostösterreich,AT


Saved NUTS-2 lookup to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\nuts2_lookup.csv
Saved NUTS-2 to NUTS-1 lookup to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\nuts2_to_nuts1_lookup.csv
Saved mapped poverty dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\poverty_social_exclusion_mapped.csv


## Download regional population data

To aggregate regional percentage indicators from NUTS-2 to NUTS-1, I need population weights.  
A simple average across NUTS-2 regions would treat small and large regions equally, which could distort the NUTS-1 value.

I therefore download Eurostat's regional population dataset. Later, I will use total population by region and year to calculate population-weighted NUTS-1 indicators.

In [69]:
# -----------------------------
# Download Eurostat regional population dataset
# -----------------------------

TARGET_DATASET = "demo_r_d2jan"

# Find dataset in Eurostat inventory
dataset_match = eurostat_inventory[
    eurostat_inventory["Code"].str.lower() == TARGET_DATASET.lower()
]

if dataset_match.empty:
    raise ValueError(f"Dataset {TARGET_DATASET} was not found in the Eurostat inventory.")

display(dataset_match.T)

download_url = dataset_match["Data download url (tsv)"].iloc[0]

print("Dataset code:", TARGET_DATASET)
print("Download URL:", download_url)

# Download raw dataset
raw_dataset_path = RAW_DIR / f"{TARGET_DATASET}.tsv"

response = requests.get(download_url, timeout=180)
response.raise_for_status()

raw_dataset_path.write_bytes(response.content)

print(f"Saved raw dataset to: {raw_dataset_path}")

# Load first preview
population_raw = pd.read_csv(raw_dataset_path, sep="\t")

population_raw.columns = population_raw.columns.str.strip()

print("Raw population dataset shape:", population_raw.shape)
display(population_raw.head())
display(population_raw.columns)

,653
Code,DEMO_R_D2JAN
Type,DATASET
Source dataset,-
Last data change,2026-05-08T11:00:00+0200
Last structural change,2026-01-20T11:00:00+0100
Data download url (tsv),https://ec.europa.eu/eurostat/api/disseminatio...
Data download url (csv),https://ec.europa.eu/eurostat/api/disseminatio...
Data download url (sdmx),https://ec.europa.eu/eurostat/api/disseminatio...
Data structure download url,https://ec.europa.eu/eurostat/api/disseminatio...
Open in Data Browser url,https://ec.europa.eu/eurostat/databrowser/prod...


Dataset code: demo_r_d2jan
Download URL: https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/DEMO_R_D2JAN/?format=TSV
Saved raw dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\raw\demo_r_d2jan.tsv
Raw population dataset shape: (160800, 37)


,"freq,unit,sex,age,geo\TIME_PERIOD",1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,"A,NR,F,TOTAL,AL",:,:,:,:,:,:,:,:,:,...,1417141,1423050,1431715,1432833,1425342,1419759,1406532,1394864,:,1194597
1,"A,NR,F,TOTAL,AL0",:,:,:,:,:,:,:,:,:,...,1417141,1423050,1431715,1432833,1425342,1419759,1406532,1394864,:,1194597
2,"A,NR,F,TOTAL,AL01",:,:,:,:,:,:,:,:,:,...,406682,405835,405598,404201,399599,396799,390886,385462,:,317141
3,"A,NR,F,TOTAL,AL02",:,:,:,:,:,:,:,:,:,...,564402,574010,585530,590623,594008,596005,597622,598531,:,506383
4,"A,NR,F,TOTAL,AL03",:,:,:,:,:,:,:,:,:,...,446057,443205,440587,438009,431735,426955,418024,410871,:,371073


Index(['freq,unit,sex,age,geo\TIME_PERIOD', '1990', '1991', '1992', '1993',
       '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002',
       '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011',
       '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020',
       '2021', '2022', '2023', '2024', '2025'],
      dtype='str')

## Prepare NUTS-2 population weights

The population dataset contains many demographic breakdowns, such as age and sex.  
For the weighted aggregation of poverty and social exclusion, I only need total population by NUTS-2 region and year.

This step filters the population data to total population, reshapes it into long format, matches it to official NUTS-2 regions, and attaches the corresponding parent NUTS-1 region.

In [70]:
# -----------------------------
# Prepare total population by NUTS-2 region and year
# -----------------------------

TARGET_DATASET = "demo_r_d2jan"
raw_dataset_path = RAW_DIR / f"{TARGET_DATASET}.tsv"

# Reload raw data if needed
population_raw = pd.read_csv(raw_dataset_path, sep="\t")
population_raw.columns = population_raw.columns.str.strip()

# Identify Eurostat combined dimension column
dimension_col = [col for col in population_raw.columns if "\\" in col][0]
dimension_names = dimension_col.split("\\")[0].split(",")

print("Dimension column:", dimension_col)
print("Detected dimensions:", dimension_names)

# Split combined dimension values into separate columns
population_split = population_raw[dimension_col].str.split(",", expand=True)
population_split.columns = dimension_names

# Clean dimension values
for col in dimension_names:
    population_split[col] = population_split[col].astype(str).str.strip()

# Attach year columns
value_cols = [col for col in population_raw.columns if col != dimension_col]

population_wide = pd.concat(
    [population_split, population_raw[value_cols]],
    axis=1
)

# Inspect available values in key dimensions before filtering
for col in ["freq", "unit", "sex", "age"]:
    if col in population_wide.columns:
        print(f"\nAvailable values for {col}:")
        display(population_wide[col].value_counts().head(20))

# -----------------------------
# Filter to total population
# -----------------------------
population_filtered = population_wide.copy()

if "freq" in population_filtered.columns:
    population_filtered = population_filtered[population_filtered["freq"] == "A"]

if "unit" in population_filtered.columns:
    population_filtered = population_filtered[population_filtered["unit"] == "NR"]

if "sex" in population_filtered.columns:
    population_filtered = population_filtered[population_filtered["sex"] == "T"]

if "age" in population_filtered.columns:
    population_filtered = population_filtered[population_filtered["age"] == "TOTAL"]

print("\nFiltered population wide shape:", population_filtered.shape)

# -----------------------------
# Reshape to long format
# -----------------------------
population_long = population_filtered.melt(
    id_vars=dimension_names,
    value_vars=value_cols,
    var_name="year",
    value_name="population_raw"
)

population_long["year"] = population_long["year"].astype(str).str.strip().astype(int)

population_long["population"] = (
    population_long["population_raw"]
    .astype(str)
    .str.extract(r"([-+]?\d*\.?\d+)")
    .astype(float)
)

# Keep only official NUTS-2 regions
population_nuts2 = population_long.merge(
    nuts2_to_nuts1[["geo", "nuts2_name", "parent_nuts1", "parent_nuts1_name", "country_code"]],
    on="geo",
    how="inner"
)

# Basic checks
print("Population NUTS-2 shape:", population_nuts2.shape)
print("Number of NUTS-2 regions:", population_nuts2["geo"].nunique())
print("Number of parent NUTS-1 regions:", population_nuts2["parent_nuts1"].nunique())
print("Year range:", population_nuts2["year"].min(), "-", population_nuts2["year"].max())

display(population_nuts2.head())

print("Missing population by year:")
display(
    population_nuts2
    .groupby("year", as_index=False)
    .agg(
        n_regions=("geo", "nunique"),
        n_missing=("population", lambda x: x.isna().sum()),
        missing_share=("population", lambda x: x.isna().mean())
    )
    .sort_values("year")
)

# Save prepared population weights
population_nuts2_path = PROCESSED_DIR / "population_nuts2_total_long.csv"
population_nuts2.to_csv(population_nuts2_path, index=False)

print(f"Saved NUTS-2 population weights to: {population_nuts2_path}")

Dimension column: freq,unit,sex,age,geo\TIME_PERIOD
Detected dimensions: ['freq', 'unit', 'sex', 'age', 'geo']

Available values for freq:


freq
A    160800
Name: count, dtype: int64


Available values for unit:


unit
NR    160800
Name: count, dtype: int64


Available values for sex:


...


Filtered population wide shape: (521, 41)
Population NUTS-2 shape: (10620, 12)
Number of NUTS-2 regions: 295
Number of parent NUTS-1 regions: 113
Year range: 1990 - 2025


,freq,unit,sex,age,geo,year,population_raw,population,nuts2_name,parent_nuts1,parent_nuts1_name,country_code
0,A,NR,T,TOTAL,AL01,1990,:,NaN,Veri,AL0,Shqipëria,AL
1,A,NR,T,TOTAL,AL02,1990,:,NaN,Qender,AL0,Shqipëria,AL
2,A,NR,T,TOTAL,AL03,1990,:,NaN,Jug,AL0,Shqipëria,AL
3,A,NR,T,TOTAL,AT11,1990,270670,270670.0,Burgenland,AT1,Ostösterreich,AT
4,A,NR,T,TOTAL,AT12,1990,1455968,1455968.0,Niederösterreich,AT1,Ostösterreich,AT


Missing population by year:


,year,n_regions,n_missing,missing_share
0,1990,295,122,0.413559
1,1991,295,92,0.311864
2,1992,295,84,0.284746
3,1993,295,84,0.284746
4,1994,295,84,0.284746
5,1995,295,83,0.281356
6,1996,295,79,0.267797
7,1997,295,79,0.267797
8,1998,295,79,0.267797
9,1999,295,76,0.257627


Saved NUTS-2 population weights to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\population_nuts2_total_long.csv


## Aggregate poverty and social exclusion to NUTS-1

The poverty and social exclusion indicator is a percentage of the population.  
Because the final AI Literacy Gap Index is built at NUTS-1 level, I aggregate NUTS-2 values to their parent NUTS-1 regions using population weights.

This avoids treating small and large NUTS-2 regions equally and creates a more representative NUTS-1 indicator.

In [71]:
# -----------------------------
# Aggregate poverty/social exclusion from NUTS-2 to NUTS-1
# -----------------------------

# Reload required datasets if needed
poverty_mapped_path = PROCESSED_DIR / "poverty_social_exclusion_mapped.csv"
population_nuts2_path = PROCESSED_DIR / "population_nuts2_total_long.csv"
nuts1_lookup_path = PROCESSED_DIR / "nuts1_lookup.csv"

if "poverty_with_mapping" not in globals():
    poverty_with_mapping = pd.read_csv(poverty_mapped_path)

if "population_nuts2" not in globals():
    population_nuts2 = pd.read_csv(population_nuts2_path)

if "nuts1_lookup" not in globals():
    nuts1_lookup = pd.read_csv(nuts1_lookup_path)

# Keep NUTS-2 poverty observations only
poverty_nuts2 = poverty_with_mapping[
    poverty_with_mapping["parent_nuts1"].notna()
].copy()

# Keep only the relevant unit if available
if "unit" in poverty_nuts2.columns:
    poverty_nuts2 = poverty_nuts2[poverty_nuts2["unit"] == "PC_POP"].copy()

# Prepare population weights
population_weights = population_nuts2[
    ["geo", "year", "population", "parent_nuts1"]
].copy()

# Merge poverty values with NUTS-2 population weights
poverty_weighted_input = poverty_nuts2.merge(
    population_weights,
    on=["geo", "year", "parent_nuts1"],
    how="left"
)

# Keep rows with both poverty value and population
poverty_weighted_input["valid_for_weighting"] = (
    poverty_weighted_input["value"].notna()
    & poverty_weighted_input["population"].notna()
    & (poverty_weighted_input["population"] > 0)
)

poverty_weighted_valid = poverty_weighted_input[
    poverty_weighted_input["valid_for_weighting"]
].copy()

poverty_weighted_valid["weighted_value"] = (
    poverty_weighted_valid["value"] * poverty_weighted_valid["population"]
)

# Total NUTS-2 population by parent NUTS-1 and year
nuts1_population_total = (
    population_weights
    .dropna(subset=["population"])
    .groupby(["parent_nuts1", "year"], as_index=False)
    .agg(
        nuts1_population_total=("population", "sum"),
        n_nuts2_population_regions=("geo", "nunique")
    )
)

# Population-weighted aggregation
poverty_nuts1_weighted = (
    poverty_weighted_valid
    .groupby(["parent_nuts1", "year"], as_index=False)
    .agg(
        poverty_social_exclusion_rate=("weighted_value", "sum"),
        population_covered=("population", "sum"),
        n_nuts2_with_poverty_data=("geo", "nunique")
    )
)

poverty_nuts1_weighted["poverty_social_exclusion_rate"] = (
    poverty_nuts1_weighted["poverty_social_exclusion_rate"]
    / poverty_nuts1_weighted["population_covered"]
)

# Add total NUTS-1 population and coverage share
poverty_nuts1_weighted = poverty_nuts1_weighted.merge(
    nuts1_population_total,
    on=["parent_nuts1", "year"],
    how="left"
)

poverty_nuts1_weighted["population_coverage_share"] = (
    poverty_nuts1_weighted["population_covered"]
    / poverty_nuts1_weighted["nuts1_population_total"]
)

# Add NUTS-1 labels
poverty_nuts1_weighted = poverty_nuts1_weighted.merge(
    nuts1_lookup[["geo", "nuts1_name", "country_code"]].rename(columns={"geo": "parent_nuts1"}),
    on="parent_nuts1",
    how="left"
)

# Rename parent NUTS-1 code to geo for later merging
poverty_nuts1_weighted = poverty_nuts1_weighted.rename(
    columns={"parent_nuts1": "geo"}
)

# Reorder columns
poverty_nuts1_weighted = poverty_nuts1_weighted[
    [
        "geo",
        "nuts1_name",
        "country_code",
        "year",
        "poverty_social_exclusion_rate",
        "population_covered",
        "nuts1_population_total",
        "population_coverage_share",
        "n_nuts2_with_poverty_data",
        "n_nuts2_population_regions"
    ]
].sort_values(["geo", "year"])

print("Weighted NUTS-1 poverty dataset shape:", poverty_nuts1_weighted.shape)
print("Number of NUTS-1 regions:", poverty_nuts1_weighted["geo"].nunique())
print("Year range:", poverty_nuts1_weighted["year"].min(), "-", poverty_nuts1_weighted["year"].max())

display(poverty_nuts1_weighted.head())

print("Coverage by year:")
display(
    poverty_nuts1_weighted
    .groupby("year", as_index=False)
    .agg(
        n_nuts1_regions=("geo", "nunique"),
        median_population_coverage=("population_coverage_share", "median"),
        min_population_coverage=("population_coverage_share", "min")
    )
    .sort_values("year")
)

# Compare with direct NUTS-1 values where available
poverty_direct_nuts1 = poverty_long.merge(
    nuts1_lookup[["geo", "nuts1_name", "country_code"]],
    on="geo",
    how="inner"
)

if "unit" in poverty_direct_nuts1.columns:
    poverty_direct_nuts1 = poverty_direct_nuts1[
        poverty_direct_nuts1["unit"] == "PC_POP"
    ].copy()

poverty_direct_nuts1 = poverty_direct_nuts1.rename(
    columns={"value": "poverty_direct_nuts1_rate"}
)[
    ["geo", "year", "poverty_direct_nuts1_rate"]
]

poverty_comparison = poverty_nuts1_weighted.merge(
    poverty_direct_nuts1,
    on=["geo", "year"],
    how="left"
)

poverty_comparison["difference_weighted_minus_direct"] = (
    poverty_comparison["poverty_social_exclusion_rate"]
    - poverty_comparison["poverty_direct_nuts1_rate"]
)

print("Comparison with direct NUTS-1 values where available:")
display(
    poverty_comparison[
        poverty_comparison["poverty_direct_nuts1_rate"].notna()
    ].head(20)
)

print("Difference summary:")
display(
    poverty_comparison["difference_weighted_minus_direct"].describe()
)

# Save final NUTS-1 poverty dataset
poverty_nuts1_path = PROCESSED_DIR / "poverty_social_exclusion_nuts1_weighted.csv"
poverty_nuts1_weighted.to_csv(poverty_nuts1_path, index=False)

poverty_comparison_path = PROCESSED_DIR / "poverty_social_exclusion_nuts1_comparison.csv"
poverty_comparison.to_csv(poverty_comparison_path, index=False)

print(f"Saved weighted NUTS-1 poverty dataset to: {poverty_nuts1_path}")
print(f"Saved direct vs weighted comparison to: {poverty_comparison_path}")

Weighted NUTS-1 poverty dataset shape: (695, 10)
Number of NUTS-1 regions: 91
Year range: 2015 - 2025


,geo,nuts1_name,country_code,year,poverty_social_exclusion_rate,population_covered,nuts1_population_total,population_coverage_share,n_nuts2_with_poverty_data,n_nuts2_population_regions
0,AL0,Shqipëria,AL,2017,58.497780,2876591.0,2876591.0,1.0,3,3
1,AL0,Shqipëria,AL,2018,53.959695,2870324.0,2870324.0,1.0,3,3
2,AL0,Shqipëria,AL,2019,50.717062,2862427.0,2862427.0,1.0,3,3
3,AL0,Shqipëria,AL,2020,46.186028,2845955.0,2845955.0,1.0,3,3
4,AL0,Shqipëria,AL,2021,46.598042,2829741.0,2829741.0,1.0,3,3


Coverage by year:


,year,n_nuts1_regions,median_population_coverage,min_population_coverage
0,2015,30,1.0,0.251362
1,2016,35,1.0,0.250093
2,2017,37,1.0,0.249654
3,2018,46,1.0,0.249140
4,2019,55,1.0,0.248327
5,2020,55,1.0,0.156511
6,2021,77,1.0,0.391562
7,2022,91,1.0,0.391500
8,2023,90,1.0,0.391709
9,2024,90,1.0,0.750155


Comparison with direct NUTS-1 values where available:


,geo,nuts1_name,country_code,year,poverty_social_exclusion_rate,population_covered,nuts1_population_total,population_coverage_share,n_nuts2_with_poverty_data,n_nuts2_population_regions,poverty_direct_nuts1_rate,difference_weighted_minus_direct


Difference summary:


count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: difference_weighted_minus_direct, dtype: float64

Saved weighted NUTS-1 poverty dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\poverty_social_exclusion_nuts1_weighted.csv
Saved direct vs weighted comparison to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\poverty_social_exclusion_nuts1_comparison.csv


## Reusable Eurostat helper functions

Several datasets in this project use the same Eurostat file structure.  
To keep the notebook reproducible and easier to follow, I define small helper functions for downloading datasets and reshaping them into tidy long format.

This avoids repeating the same parsing logic for every dataset and makes the data collection process easier to maintain.

In [72]:
# -----------------------------
# Reusable Eurostat helper functions
# -----------------------------

def get_eurostat_dataset_info(dataset_code, inventory):
    """
    Find a Eurostat dataset in the downloaded inventory.
    """
    dataset_match = inventory[
        inventory["Code"].str.lower() == dataset_code.lower()
    ]

    if dataset_match.empty:
        raise ValueError(f"Dataset {dataset_code} was not found in the Eurostat inventory.")

    return dataset_match.iloc[0]


def download_eurostat_tsv(dataset_code, inventory, raw_dir, overwrite=False):
    """
    Download a Eurostat TSV dataset from the official inventory URL.
    Saves the raw file to data/raw and returns the local path.
    """
    dataset_info = get_eurostat_dataset_info(dataset_code, inventory)
    download_url = dataset_info["Data download url (tsv)"]

    raw_path = raw_dir / f"{dataset_code}.tsv"

    if raw_path.exists() and not overwrite:
        print(f"{dataset_code}: raw file already exists at {raw_path}")
        return raw_path

    response = requests.get(download_url, timeout=180)
    response.raise_for_status()

    raw_path.write_bytes(response.content)

    print(f"{dataset_code}: saved raw dataset to {raw_path}")
    return raw_path


def reshape_eurostat_tsv(raw_path, value_name="value"):
    """
    Reshape a Eurostat TSV file from wide format into tidy long format.
    Returns one row per dimension combination and year.
    """
    raw = pd.read_csv(raw_path, sep="\t")
    raw.columns = raw.columns.str.strip()

    dimension_cols = [col for col in raw.columns if "\\" in col]

    if len(dimension_cols) != 1:
        raise ValueError(
            f"Expected exactly one Eurostat dimension column, found {len(dimension_cols)}."
        )

    dimension_col = dimension_cols[0]
    dimension_names = dimension_col.split("\\")[0].split(",")

    split_dimensions = raw[dimension_col].str.split(",", expand=True)
    split_dimensions.columns = dimension_names

    for col in dimension_names:
        split_dimensions[col] = split_dimensions[col].astype(str).str.strip()

    value_cols = [col for col in raw.columns if col != dimension_col]

    wide = pd.concat(
        [split_dimensions, raw[value_cols]],
        axis=1
    )

    long = wide.melt(
        id_vars=dimension_names,
        value_vars=value_cols,
        var_name="year",
        value_name=f"{value_name}_raw"
    )

    long["year"] = long["year"].astype(str).str.strip().astype(int)

    long[value_name] = (
        long[f"{value_name}_raw"]
        .astype(str)
        .str.extract(r"([-+]?\d*\.?\d+)")
        .astype(float)
    )

    return long


def summarize_tidy_dataset(df, dataset_name, value_col="value"):
    """
    Print a compact summary of a tidy dataset.
    """
    print(f"Dataset: {dataset_name}")
    print("Shape:", df.shape)

    if "year" in df.columns:
        print("Year range:", df["year"].min(), "-", df["year"].max())

    if "geo" in df.columns:
        print("Geo codes:", df["geo"].nunique())

    print("\nColumns:")
    display(df.columns.to_list())

    print("\nMissing values by year:")
    if "year" in df.columns:
        display(
            df
            .groupby("year", as_index=False)
            .agg(
                n_rows=(value_col, "size"),
                n_non_missing=(value_col, "count"),
                n_missing=(value_col, lambda x: x.isna().sum()),
                missing_share=(value_col, lambda x: x.isna().mean())
            )
            .sort_values("year")
        )

    display(df.head())

## Download remaining candidate datasets

To complete the data collection notebook, I download the remaining candidate datasets for the AI Literacy Gap Index.  
At this stage, the datasets are not merged yet. Each dataset is saved separately in tidy long format so that the following EDA notebook can inspect coverage, missing values, regional levels, and conceptual usefulness before final feature selection.

In [73]:
# -----------------------------
# Download and reshape remaining candidate datasets
# -----------------------------

remaining_datasets = [
    {
        "dataset_code": "edat_lfse_04",
        "save_name": "education_attainment",
        "pillar": "Education",
        "description": "Educational attainment by region"
    },
    {
        "dataset_code": "trng_lfse_04",
        "save_name": "lifelong_learning",
        "pillar": "Adult learning",
        "description": "Participation in education and training by region"
    },
    {
        "dataset_code": "lfst_r_lfu3pers",
        "save_name": "unemployment_education_region",
        "pillar": "Labour market vulnerability",
        "description": "Unemployment by education level and region"
    },
    {
        "dataset_code": "isoc_r_eb_ain2",
        "save_name": "enterprise_ai_adoption",
        "pillar": "AI exposure",
        "description": "Enterprise AI adoption by region"
    }
]

download_results = []

for dataset in remaining_datasets:
    dataset_code = dataset["dataset_code"]
    save_name = dataset["save_name"]

    print("=" * 80)
    print(f"Processing {dataset_code}: {dataset['description']}")

    try:
        raw_path = download_eurostat_tsv(
            dataset_code=dataset_code,
            inventory=eurostat_inventory,
            raw_dir=RAW_DIR,
            overwrite=False
        )

        tidy_df = reshape_eurostat_tsv(
            raw_path=raw_path,
            value_name="value"
        )

        output_path = PROCESSED_DIR / f"{save_name}_tidy.csv"
        tidy_df.to_csv(output_path, index=False)

        summarize_tidy_dataset(
            df=tidy_df,
            dataset_name=dataset_code,
            value_col="value"
        )

        download_results.append({
            "dataset_code": dataset_code,
            "save_name": save_name,
            "pillar": dataset["pillar"],
            "description": dataset["description"],
            "status": "success",
            "raw_path": raw_path,
            "processed_path": output_path,
            "n_rows": tidy_df.shape[0],
            "n_columns": tidy_df.shape[1],
            "n_geo_codes": tidy_df["geo"].nunique() if "geo" in tidy_df.columns else None,
            "min_year": tidy_df["year"].min() if "year" in tidy_df.columns else None,
            "max_year": tidy_df["year"].max() if "year" in tidy_df.columns else None
        })

    except Exception as e:
        print(f"Could not process {dataset_code}: {e}")

        download_results.append({
            "dataset_code": dataset_code,
            "save_name": save_name,
            "pillar": dataset["pillar"],
            "description": dataset["description"],
            "status": "failed",
            "raw_path": None,
            "processed_path": None,
            "n_rows": None,
            "n_columns": None,
            "n_geo_codes": None,
            "min_year": None,
            "max_year": None,
            "error": str(e)
        })

download_summary = pd.DataFrame(download_results)

display(download_summary)

download_summary_path = PROCESSED_DIR / "download_summary_remaining_datasets.csv"
download_summary.to_csv(download_summary_path, index=False)

print(f"Saved download summary to: {download_summary_path}")

Processing edat_lfse_04: Educational attainment by region
edat_lfse_04: saved raw dataset to c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\raw\edat_lfse_04.tsv
Dataset: edat_lfse_04
Shape: (939900, 9)
Year range: 2000 - 2025
Geo codes: 511

Columns:


['freq', 'sex', 'isced11', 'age', 'unit', 'geo', 'year', 'value_raw', 'value']


Missing values by year:


,year,n_rows,n_non_missing,n_missing,missing_share
0,2000,36150,17468,18682,0.516791
1,2001,36150,17910,18240,0.504564
2,2002,36150,18264,17886,0.494772
3,2003,36150,18673,17477,0.483458
4,2004,36150,18748,17402,0.481383
5,2005,36150,19106,17044,0.471480
6,2006,36150,21159,14991,0.414689
7,2007,36150,21470,14680,0.406086
8,2008,36150,21474,14676,0.405975
9,2009,36150,21491,14659,0.405505


,freq,sex,isced11,age,unit,geo,year,value_raw,value
0,A,F,ED0-2,Y20-24,PC,AT,2000,15.1,15.1
1,A,F,ED0-2,Y20-24,PC,AT1,2000,15.4,15.4
2,A,F,ED0-2,Y20-24,PC,AT11,2000,: u,NaN
3,A,F,ED0-2,Y20-24,PC,AT12,2000,16.3,16.3
4,A,F,ED0-2,Y20-24,PC,AT13,2000,14.4,14.4


...


Missing values by year:


,year,n_rows,n_non_missing,n_missing,missing_share
0,2023,58910,35374,23536,0.399525
1,2024,58910,31926,26984,0.458055
2,2025,58910,48538,10372,0.176065


,freq,nace_r2,size_emp,indic_is,unit,geo,year,value_raw,value
0,A,C,GE10,E_AI_CC,PC_ENT,AT,2023,9.95,9.95
1,A,C,GE10,E_AI_CC,PC_ENT,AT1,2023,9.35,9.35
2,A,C,GE10,E_AI_CC,PC_ENT,AT11,2023,5.81,5.81
3,A,C,GE10,E_AI_CC,PC_ENT,AT12,2023,6.38,6.38
4,A,C,GE10,E_AI_CC,PC_ENT,AT13,2023,17.93,17.93


,dataset_code,save_name,pillar,description,status,raw_path,processed_path,n_rows,n_columns,n_geo_codes,min_year,max_year
0,edat_lfse_04,education_attainment,Education,Educational attainment by region,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,939900,9,511,2000,2025
1,trng_lfse_04,lifelong_learning,Adult learning,Participation in education and training by region,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,79716,8,511,2000,2025
2,lfst_r_lfu3pers,unemployment_education_region,Labour market vulnerability,Unemployment by education level and region,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,1870155,9,512,1999,2025
3,isoc_r_eb_ain2,enterprise_ai_adoption,AI exposure,Enterprise AI adoption by region,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,176730,9,179,2023,2025


Saved download summary to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\download_summary_remaining_datasets.csv


## Add readable labels to the downloaded datasets

Most Eurostat datasets use technical codes for dimensions such as region, age group, education level, unit, sex, and indicator type.  
To make the datasets easier to understand for the next EDA step, I enrich the tidy datasets with readable labels from the Eurostat metadata API.

The original code columns are kept, while additional label columns are added next to them. This keeps the data both machine-readable and human-readable.

In [74]:
# -----------------------------
# Helper functions for Eurostat labels
# -----------------------------

def get_eurostat_metadata(dataset_code, latest_year=None):
    """
    Download Eurostat JSON metadata for a dataset.
    A latest_year filter is used where possible to keep the API response smaller.
    """
    metadata_url = (
        f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
        f"{dataset_code}"
    )

    params = {"lang": "en"}

    if latest_year is not None:
        params["time"] = str(latest_year)

    response = requests.get(metadata_url, params=params, timeout=180)

    # Fallback: retry without time filter if the filtered request fails
    if response.status_code != 200 and latest_year is not None:
        response = requests.get(metadata_url, params={"lang": "en"}, timeout=180)

    response.raise_for_status()
    return response.json()


def extract_dimension_labels(metadata, dimension_name):
    """
    Extract code-label mappings for one Eurostat dimension.
    """
    if "dimension" not in metadata or dimension_name not in metadata["dimension"]:
        return None

    dimension = metadata["dimension"][dimension_name]
    category = dimension.get("category", {})

    labels = category.get("label", {})
    index = category.get("index", {})

    if isinstance(index, dict) and len(index) > 0:
        codes = list(index.keys())
    else:
        codes = list(labels.keys())

    label_df = pd.DataFrame({
        dimension_name: codes,
        f"{dimension_name}_label": [labels.get(code, code) for code in codes]
    })

    return label_df


def add_eurostat_labels(df, dataset_code):
    """
    Add readable Eurostat labels for all matching dimensions in a tidy dataframe.
    """
    latest_year = int(df["year"].max()) if "year" in df.columns else None
    metadata = get_eurostat_metadata(dataset_code, latest_year=latest_year)

    labeled_df = df.copy()

    dimensions_to_try = [
        col for col in labeled_df.columns
        if col not in ["year", "value", "value_raw"]
        and not col.endswith("_label")
    ]

    added_labels = []

    for dimension in dimensions_to_try:
        label_df = extract_dimension_labels(metadata, dimension)

        if label_df is not None and dimension in labeled_df.columns:
            labeled_df = labeled_df.merge(label_df, on=dimension, how="left")
            added_labels.append(f"{dimension}_label")

    print(f"{dataset_code}: added label columns:")
    display(added_labels)

    return labeled_df


# -----------------------------
# Add labels to all relevant tidy datasets
# -----------------------------

datasets_to_label = [
    {
        "dataset_code": "tgs00107",
        "input_file": "poverty_social_exclusion_tidy.csv",
        "output_file": "poverty_social_exclusion_labeled.csv"
    },
    {
        "dataset_code": "edat_lfse_04",
        "input_file": "education_attainment_tidy.csv",
        "output_file": "education_attainment_labeled.csv"
    },
    {
        "dataset_code": "trng_lfse_04",
        "input_file": "lifelong_learning_tidy.csv",
        "output_file": "lifelong_learning_labeled.csv"
    },
    {
        "dataset_code": "lfst_r_lfu3pers",
        "input_file": "unemployment_education_region_tidy.csv",
        "output_file": "unemployment_education_region_labeled.csv"
    },
    {
        "dataset_code": "isoc_r_eb_ain2",
        "input_file": "enterprise_ai_adoption_tidy.csv",
        "output_file": "enterprise_ai_adoption_labeled.csv"
    }
]

labeling_results = []

for item in datasets_to_label:
    dataset_code = item["dataset_code"]
    input_path = PROCESSED_DIR / item["input_file"]
    output_path = PROCESSED_DIR / item["output_file"]

    print("=" * 80)
    print(f"Labeling dataset: {dataset_code}")

    if not input_path.exists():
        print(f"Input file not found, skipping: {input_path}")

        labeling_results.append({
            "dataset_code": dataset_code,
            "status": "skipped_missing_input",
            "input_path": input_path,
            "output_path": output_path
        })

        continue

    try:
        tidy_df = pd.read_csv(input_path)
        labeled_df = add_eurostat_labels(tidy_df, dataset_code)

        labeled_df.to_csv(output_path, index=False)

        print("Saved labeled dataset to:", output_path)
        display(labeled_df.head())

        labeling_results.append({
            "dataset_code": dataset_code,
            "status": "success",
            "input_path": input_path,
            "output_path": output_path,
            "n_rows": labeled_df.shape[0],
            "n_columns": labeled_df.shape[1],
            "n_geo_codes": labeled_df["geo"].nunique() if "geo" in labeled_df.columns else None,
            "min_year": labeled_df["year"].min() if "year" in labeled_df.columns else None,
            "max_year": labeled_df["year"].max() if "year" in labeled_df.columns else None
        })

    except Exception as e:
        print(f"Could not label {dataset_code}: {e}")

        labeling_results.append({
            "dataset_code": dataset_code,
            "status": "failed",
            "input_path": input_path,
            "output_path": output_path,
            "error": str(e)
        })

labeling_summary = pd.DataFrame(labeling_results)

display(labeling_summary)

labeling_summary_path = PROCESSED_DIR / "labeling_summary.csv"
labeling_summary.to_csv(labeling_summary_path, index=False)

print(f"Saved labeling summary to: {labeling_summary_path}")

Labeling dataset: tgs00107
tgs00107: added label columns:


['freq_label', 'unit_label', 'geo_label']

Saved labeled dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\poverty_social_exclusion_labeled.csv


,freq,unit,geo,year,value_raw,value,freq_label,unit_label,geo_label
0,A,PC_POP,AL01,2015,:,NaN,Annual,Percentage of total population,Veri
1,A,PC_POP,AL02,2015,:,NaN,Annual,Percentage of total population,Qender
2,A,PC_POP,AL03,2015,:,NaN,Annual,Percentage of total population,Jug
3,A,PC_POP,AT11,2015,:,NaN,Annual,Percentage of total population,Burgenland
4,A,PC_POP,AT12,2015,:,NaN,Annual,Percentage of total population,Niederösterreich


Labeling dataset: edat_lfse_04
edat_lfse_04: added label columns:


...

['freq_label',
 'nace_r2_label',
 'size_emp_label',
 'indic_is_label',
 'unit_label',
 'geo_label']

Saved labeled dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\enterprise_ai_adoption_labeled.csv


,freq,nace_r2,size_emp,indic_is,unit,geo,year,value_raw,value,freq_label,nace_r2_label,size_emp_label,indic_is_label,unit_label,geo_label
0,A,C,GE10,E_AI_CC,PC_ENT,AT,2023,9.95,9.95,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Austria
1,A,C,GE10,E_AI_CC,PC_ENT,AT1,2023,9.35,9.35,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Ostösterreich
2,A,C,GE10,E_AI_CC,PC_ENT,AT11,2023,5.81,5.81,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Burgenland
3,A,C,GE10,E_AI_CC,PC_ENT,AT12,2023,6.38,6.38,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Niederösterreich
4,A,C,GE10,E_AI_CC,PC_ENT,AT13,2023,17.93,17.93,Annual,Manufacturing,10 persons employed or more,Enterprises use AI technologies and buy any cl...,Percentage of enterprises,Wien


,dataset_code,status,input_path,output_path,n_rows,n_columns,n_geo_codes,min_year,max_year
0,tgs00107,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,2959,9,269,2015,2025
1,edat_lfse_04,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,939900,15,511,2000,2025
2,trng_lfse_04,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,79716,13,511,2000,2025
3,lfst_r_lfu3pers,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,1870155,15,512,1999,2025
4,isoc_r_eb_ain2,success,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,176730,15,179,2023,2025


Saved labeling summary to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\labeling_summary.csv


## Create data catalog for EDA handover

To make the notebook useful for the next project phase, I create a compact data catalog of all processed datasets.  
The catalog summarizes each dataset's role in the AI Literacy Gap Index, its file location, time coverage, regional coverage, and basic missing-value structure.

This makes it easier for the next EDA notebook to decide which datasets are ready to inspect, merge, or filter further.

In [75]:
# -----------------------------
# Create data catalog for EDA handover
# -----------------------------

# Reload lookup tables if needed
nuts1_lookup_path = PROCESSED_DIR / "nuts1_lookup.csv"
nuts2_lookup_path = PROCESSED_DIR / "nuts2_lookup.csv"

nuts1_lookup = pd.read_csv(nuts1_lookup_path) if nuts1_lookup_path.exists() else pd.DataFrame()
nuts2_lookup = pd.read_csv(nuts2_lookup_path) if nuts2_lookup_path.exists() else pd.DataFrame()

nuts1_codes = set(nuts1_lookup["geo"]) if "geo" in nuts1_lookup.columns else set()
nuts2_codes = set(nuts2_lookup["geo"]) if "geo" in nuts2_lookup.columns else set()


def detect_geo_coverage(df):
    """
    Detects whether geo codes in a dataset correspond to NUTS-1, NUTS-2, country-level,
    or unmatched codes.
    """
    if "geo" not in df.columns:
        return {
            "n_geo_codes": None,
            "n_nuts1_codes": None,
            "n_nuts2_codes": None,
            "n_country_like_codes": None,
            "n_other_geo_codes": None
        }

    geo_codes = set(df["geo"].dropna().astype(str).unique())

    n_nuts1 = len(geo_codes.intersection(nuts1_codes))
    n_nuts2 = len(geo_codes.intersection(nuts2_codes))
    n_country_like = sum(len(code) == 2 for code in geo_codes)
    n_other = len(geo_codes) - len(
        geo_codes.intersection(nuts1_codes).union(geo_codes.intersection(nuts2_codes))
    )

    return {
        "n_geo_codes": len(geo_codes),
        "n_nuts1_codes": n_nuts1,
        "n_nuts2_codes": n_nuts2,
        "n_country_like_codes": n_country_like,
        "n_other_geo_codes": n_other
    }


project_datasets = [
    {
        "dataset_code": "isoc_r_dskl_i",
        "dataset_name": "Digital skills by NUTS-1 region",
        "pillar": "Digital readiness",
        "processed_file": "digital_skills_nuts1_labeled_filtered.csv",
        "main_use": "Core proxy for digital readiness / AI literacy readiness"
    },
    {
        "dataset_code": "tgs00107",
        "dataset_name": "People at risk of poverty or social exclusion",
        "pillar": "Social vulnerability",
        "processed_file": "poverty_social_exclusion_nuts1_weighted.csv",
        "main_use": "Population-weighted NUTS-1 vulnerability indicator"
    },
    {
        "dataset_code": "demo_r_d2jan",
        "dataset_name": "Population by age, sex and region",
        "pillar": "Population weights / demographics",
        "processed_file": "population_nuts2_total_long.csv",
        "main_use": "Population weights for NUTS-2 to NUTS-1 aggregation"
    },
    {
        "dataset_code": "edat_lfse_04",
        "dataset_name": "Educational attainment by region",
        "pillar": "Education",
        "processed_file": "education_attainment_labeled.csv",
        "main_use": "Education structure as AI literacy readiness factor"
    },
    {
        "dataset_code": "trng_lfse_04",
        "dataset_name": "Participation in education and training",
        "pillar": "Adult learning",
        "processed_file": "lifelong_learning_labeled.csv",
        "main_use": "Lifelong learning / reskilling capacity"
    },
    {
        "dataset_code": "lfst_r_lfu3pers",
        "dataset_name": "Unemployment by education level and region",
        "pillar": "Labour market vulnerability",
        "processed_file": "unemployment_education_region_labeled.csv",
        "main_use": "Labour market risk by education level"
    },
    {
        "dataset_code": "isoc_r_eb_ain2",
        "dataset_name": "Enterprise AI adoption by region",
        "pillar": "AI exposure",
        "processed_file": "enterprise_ai_adoption_labeled.csv",
        "main_use": "Regional AI adoption pressure from enterprises"
    }
]

catalog_rows = []

for item in project_datasets:
    path = PROCESSED_DIR / item["processed_file"]

    row = item.copy()
    row["file_exists"] = path.exists()
    row["processed_path"] = str(path)

    if path.exists():
        df = pd.read_csv(path)

        row["n_rows"] = df.shape[0]
        row["n_columns"] = df.shape[1]

        if "year" in df.columns:
            row["min_year"] = int(df["year"].min())
            row["max_year"] = int(df["year"].max())
            row["n_years"] = df["year"].nunique()
        else:
            row["min_year"] = None
            row["max_year"] = None
            row["n_years"] = None

        value_cols = [col for col in df.columns if col in ["value", "population", "poverty_social_exclusion_rate"]]
        if value_cols:
            value_col = value_cols[0]
            row["main_value_column"] = value_col
            row["missing_share_main_value"] = df[value_col].isna().mean()
        else:
            row["main_value_column"] = None
            row["missing_share_main_value"] = None

        row.update(detect_geo_coverage(df))

        row["columns"] = ", ".join(df.columns.astype(str))
    else:
        row["n_rows"] = None
        row["n_columns"] = None
        row["min_year"] = None
        row["max_year"] = None
        row["n_years"] = None
        row["main_value_column"] = None
        row["missing_share_main_value"] = None
        row["n_geo_codes"] = None
        row["n_nuts1_codes"] = None
        row["n_nuts2_codes"] = None
        row["n_country_like_codes"] = None
        row["n_other_geo_codes"] = None
        row["columns"] = None

    catalog_rows.append(row)

data_catalog = pd.DataFrame(catalog_rows)

display(data_catalog)

data_catalog_path = PROCESSED_DIR / "project_data_catalog.csv"
data_catalog.to_csv(data_catalog_path, index=False)

print(f"Saved project data catalog to: {data_catalog_path}")

,dataset_code,dataset_name,pillar,processed_file,main_use,file_exists,processed_path,n_rows,n_columns,min_year,max_year,n_years,main_value_column,missing_share_main_value,n_geo_codes,n_nuts1_codes,n_nuts2_codes,n_country_like_codes,n_other_geo_codes,columns
0,isoc_r_dskl_i,Digital skills by NUTS-1 region,Digital readiness,digital_skills_nuts1_labeled_filtered.csv,Core proxy for digital readiness / AI literacy...,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,4126,13,2025,2025,1,value,0.005817,88,88,0,0,0,"freq, indic_is, unit, geo, year, value_raw, va..."
1,tgs00107,People at risk of poverty or social exclusion,Social vulnerability,poverty_social_exclusion_nuts1_weighted.csv,Population-weighted NUTS-1 vulnerability indic...,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,695,10,2015,2025,11,poverty_social_exclusion_rate,0.000000,91,91,0,0,0,"geo, nuts1_name, country_code, year, poverty_s..."
2,demo_r_d2jan,"Population by age, sex and region",Population weights / demographics,population_nuts2_total_long.csv,Population weights for NUTS-2 to NUTS-1 aggreg...,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,10620,12,1990,2025,36,population,0.155179,295,0,295,0,0,"freq, unit, sex, age, geo, year, population_ra..."
3,edat_lfse_04,Educational attainment by region,Education,education_attainment_labeled.csv,Education structure as AI literacy readiness f...,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,939900,15,2000,2025,26,value,0.317754,511,111,289,36,111,"freq, sex, isced11, age, unit, geo, year, valu..."
4,trng_lfse_04,Participation in education and training,Adult learning,lifelong_learning_labeled.csv,Lifelong learning / reskilling capacity,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,79716,13,2000,2025,26,value,0.144237,511,111,289,36,111,"freq, unit, sex, age, geo, year, value_raw, va..."
5,lfst_r_lfu3pers,Unemployment by education level and region,Labour market vulnerability,unemployment_education_region_labeled.csv,Labour market risk by education level,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,1870155,15,1999,2025,27,value,0.485299,512,111,290,36,111,"freq, isced11, sex, age, unit, geo, year, valu..."
6,isoc_r_eb_ain2,Enterprise AI adoption by region,AI exposure,enterprise_ai_adoption_labeled.csv,Regional AI adoption pressure from enterprises,True,c:\Users\Lu\OneDrive\ToU\chl_data_science_proj...,176730,15,2023,2025,3,value,0.344548,179,38,106,34,35,"freq, nace_r2, size_emp, indic_is, unit, geo, ..."


Saved project data catalog to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\project_data_catalog.csv
